# Reservoir Water Quality — SQL Analysis
Coursework from **Programming for Geographic Data Science**, NTU Geography (Spring 2023).
Taiwan's reservoir water quality readings loaded into SQLite, flagged for completeness,
and queried in SQL.

Data: `reservoir_water_quality.csv`, a snapshot from the Ministry of Environment's open
data platform (samples 2026-01-06 to 2026-07-01). The assignment read the same dataset
from an API, which moved domains when the agency was reorganised; the snapshot keeps this
reproducible.
 
# 1. Load the data into SQLite

Readings arrive as text. `'-'` means not recorded and is stored as `NULL`; a `'<'`
prefix (`'<4.0'`, below the detection limit) is stripped and stored as a number.
A `chk` flag marks each row `'T'` or `'F'` so completeness is queryable.


In [11]:
import csv
import sqlite3
 
def clean_value(raw):
    """Return a float, or None when the value is not recorded."""
    if raw is None:
        return None
    text = str(raw).strip()
    if text in ('', '-'):
        return None
    text = text.replace('<', '')        # '<4.0' -> '4.0'
    try:
        return float(text)
    except ValueError:
        return None
 
 
conn = sqlite3.connect('WQ.db')
cur = conn.cursor()
 
cur.execute("""CREATE TABLE IF NOT EXISTS Flow
               (sitename TEXT, damname TEXT, county TEXT, sampledate TEXT,
                itemname TEXT, itemengabbreviation TEXT, itemvalue NUMERIC,
                itemunit TEXT, chk TEXT)""")
cur.execute('DELETE FROM Flow')         # rebuild on every run, no duplicates
 
with open('reservoir_water_quality.csv', encoding='utf-8-sig', newline='') as f:
    for row in csv.DictReader(f):
        value = clean_value(row['itemvalue'])
        cur.execute('INSERT INTO Flow VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)',
                    (row['sitename'], row['damname'], row['county'], row['sampledate'],
                     row['itemname'], row['itemengabbreviation'], value,
                     row['itemunit'], 'T' if value is not None else 'F'))
        
conn.commit()
print(cur.execute('SELECT COUNT(*) FROM Flow').fetchone()[0], 'rows loaded')


5775 rows loaded


# 2. What is in the table
A row is one measurement of one parameter at one station on one sample date;
`damname` groups stations. Stations are counted as `damname`/`sitename` pairs, so a
name reused at two reservoirs would not collapse into one.

In [9]:
for label, sql in [
    ('rows                ', "SELECT COUNT(*) FROM Flow"),
    ('reservoirs          ', "SELECT COUNT(DISTINCT damname) FROM Flow"),
    ('stations            ', "SELECT COUNT(DISTINCT damname || '|' || sitename) FROM Flow"),
    ('parameters          ', "SELECT COUNT(DISTINCT itemengabbreviation) FROM Flow"),
    ('counties            ', "SELECT COUNT(DISTINCT county) FROM Flow"),
    ('rows with no value  ', "SELECT COUNT(*) FROM Flow WHERE chk = 'F'"),
    ('share with no value ', "SELECT ROUND(SUM(CASE WHEN chk = 'F' THEN 1.0 ELSE 0 END)"
                             " / COUNT(*), 3) FROM Flow"),
]:
    print(label, cur.execute(sql).fetchone()[0])

rows                 5775
reservoirs           26
stations             85
parameters           25
counties             13
rows with no value   2436
share with no value  0.422


# Where the missing values are

42% of rows carry no value — worth locating before trusting any average from this table.

In [3]:
rows = cur.execute("""
    SELECT itemengabbreviation,
           MAX(itemname),
           COUNT(*),
           SUM(CASE WHEN chk = 'F' THEN 1 ELSE 0 END),
           ROUND(SUM(CASE WHEN chk = 'F' THEN 1.0 ELSE 0 END) / COUNT(*), 3)
    FROM Flow
    GROUP BY itemengabbreviation
    ORDER BY 5 DESC
""").fetchall()
 
print(f"{'parameter':<12}{'name':<16}{'rows':>6}{'missing':>9}{'rate':>8}")
for eng, name, total, miss, rate in rows:
    print(f'{eng:<12}{name:<16}{total:>6}{miss:>9}{rate:>8.1%}')
 

parameter   name              rows  missing    rate
TKN         總凱氏氮               231      231  100.0%
TH          總硬度                231      231  100.0%
TAlk        總鹼度                231      231  100.0%
PO4         正磷酸鹽               231      231  100.0%
Norg        有機氮                231      231  100.0%
NO3_N       硝酸鹽氮               231      231  100.0%
NO2_N       亞硝酸鹽氮              231      231  100.0%
DO2         溶氧(滴定法)            231      231  100.0%
SD          透明度                231       64   27.7%
CTSI        卡爾森指數              231       64   27.7%
TOC         總有機碳               231       58   25.1%
TN          總氮                 231       58   25.1%
SS          懸浮固體               231       52   22.5%
DOS         溶氧飽和度              231       52   22.5%
pH          酸鹼值                231       22    9.5%
WT          水溫                 231       22    9.5%
TP          總磷                 231       22    9.5%
TB          濁度                 231       22    9.5%
SampleDepth 

In [4]:
def warning_site(ItemEng, threshold):
    data = cur.execute("""
        SELECT sitename, damname, itemvalue
        FROM Flow
        WHERE itemengabbreviation = ? AND chk = 'T' AND itemvalue > ?
    """, (ItemEng, threshold)).fetchall()
    if not data:
        print('查無資料')
    else:
        for record in data:
            print(record)
warning_site('Chl_a', 100)

('鳳山水庫二', '鳳山水庫', 143)
('鳳山水庫三', '鳳山水庫', 155)
('鳳山水庫四', '鳳山水庫', 107)
('鳳山水庫四', '鳳山水庫', 149)
('鳳山水庫二', '鳳山水庫', 119)
('鳳山水庫三', '鳳山水庫', 132)
('鳳山水庫四', '鳳山水庫', 130)


`query_wq` — station count, parameter name, mean and unit for one reservoir
 
`AVG` ignores `NULL`, so missing rows need no `WHERE` filter. Filtering them out would
also drop the station count and unit, which should report even when nothing was
measured — `MAX` on the text columns keeps them.

In [5]:
def query_wq(Site, ItemEng):
    site_count, item_name, avg_value, item_unit = cur.execute("""
        SELECT COUNT(DISTINCT sitename), MAX(itemname), AVG(itemvalue), MAX(itemunit)
        FROM Flow
        WHERE damname = ? AND itemengabbreviation = ?
    """, (Site, ItemEng)).fetchone()
    if avg_value is not None:
        avg_value = round(avg_value, 3)
    print((site_count, item_name, avg_value, item_unit))
 
 
query_wq(Site='翡翠水庫', ItemEng='pH')
query_wq(Site='翡翠水庫', ItemEng='DO2')     # nothing recorded for this parameter
 

 

(6, '酸鹼值', 8.013, ' ')
(6, '溶氧(滴定法)', None, 'mg/L')


### `check_data` — the share of a reservoir's stations with no reading at all

A station can be sampled more than once, so counting rows would overstate the share.
`CASE WHEN` returns `NULL` when the condition fails and `COUNT` ignores `NULL`, so
subtracting the stations that have a reading leaves those that have none.

The division is done in Python: in SQLite `SUM(...) / COUNT(...)` on two integers is
integer division and returns 0 for every reservoir.

In [6]:
def check_data(Site, ItemEng):
    site_count, null_site_count = cur.execute("""
        SELECT COUNT(DISTINCT sitename),
               COUNT(DISTINCT sitename)
               - COUNT(DISTINCT CASE WHEN chk = 'T' THEN sitename END)
        FROM Flow
        WHERE damname = ? AND itemengabbreviation = ?
    """, (Site, ItemEng)).fetchone()
    if site_count == 0:
        print('查無資料')
        return
    print(round(null_site_count / site_count, 3))
 
 
check_data(Site='翡翠水庫', ItemEng='pH')
check_data(Site='翡翠水庫', ItemEng='DO2')
check_data(Site='不存在的水庫', ItemEng='pH')

0.0
1.0
查無資料


In [7]:
conn.close()